In [1]:
"""
Human → Comic Face GAN  v6
==========================
Targeted improvements over v4 (avoiding v5's broken additions):

NEW IN v6:
  ① DiffAugment (color jitter + translation + cutout) — prevents D memorization
     and D saturation.  Applied to BOTH real and fake before D forward pass.
  ② Adaptive D training (D:G step ratio 1:1 → 2:1 when D is weak, 1:2 when saturated).
     Keeps D in the "useful zone" (loss 0.4–0.7). Solves the 0.3252 plateau.
  ③ Per-stage bottleneck rebuild — n_res_blocks is now actually per-stage
     (was always max(6) regardless of depth in v4, wasting capacity at 64px).
  ④ Stage 1 anti-overfitting: cosine LR with aggressive warmup + weight decay
     on G optimizer (1e-4). Val G diverged from train after ep21 → regularize.
  ⑤ Noise injection into D inputs (σ=0.05 annealed to 0 over training).
     Soft D target helps prevent D from becoming too confident too fast.
  ⑥ Frequency loss (FFT magnitude L1) — pushes G to match high-freq comic
     edges/lines that L1 and perceptual loss both under-weight.
  ⑦ Self-attention added at encoder level 3 (nf*4) in addition to bottleneck
     — helps preserve structural identity across the translation.
  ⑧ Visualization saves to PNG files (no plt.show() blocking in Kaggle).
  ⑨ Stage-aware gradient accumulation: acc=1 for 64/128, acc=4 for 256
     → effective bs=4 even at full resolution.
  ⑩ EMA shadow depth synced automatically in set_depth().

All v4 stability fixes retained:
  - GroupNorm, Dual GradScaler, grad clipping, NaN guard
  - Label smoothing, EMA, progressive 64→128→256
  - Best-checkpoint saving, CBAM skip gates
"""

import copy, math, os, warnings, random
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib
matplotlib.use('Agg')          # non-interactive backend
import matplotlib.pyplot as plt
from tqdm import tqdm
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torch.cuda.amp import autocast, GradScaler
from transformers import get_cosine_schedule_with_warmup

# ─────────────────────────────────────────────────────────────────────
#  CONFIGURATION
# ─────────────────────────────────────────────────────────────────────
H = {
    'debug': False,

    # (resolution, epochs, batch_size, grad_accum_steps)
    'stages': [
        (64,  50, 8, 1),
        (128, 30, 4, 1),
        (256, 20, 2, 4),   # effective bs=8 at 256px
    ],

    'lr':            2e-4,
    'beta1':         0.5,
    'weight_decay':  1e-4,   # ← NEW: L2 on G to reduce stage-1 overfitting
    'max_grad_norm': 1.0,

    'lambda_l1':    10,
    'lambda_perc':   5,
    'lambda_fm':     2,
    'lambda_freq':   1,      # ← NEW: FFT frequency loss weight

    'ndf':          64,
    'ngf':          64,
    'dropout':       0.3,
    'n_layers_D':    3,

    # Residual blocks tuned per stage (fixed from v4 where it was always max)
    'n_res_blocks': {64: 2, 128: 4, 256: 6},

    'real_label':  0.9,
    'fake_label':  0.1,
    'ema_decay':   0.999,
    'save_dir':    './',

    # ── Adaptive D training ──────────────────────────────────────
    # When val D loss < d_low_thresh  → skip 1 D step per 2 G steps
    # When val D loss > d_high_thresh → do 2 D steps per 1 G step
    'd_low_thresh':  0.35,
    'd_high_thresh': 0.70,

    # ── D input noise (annealed to 0 over all steps) ─────────────
    'd_noise_sigma': 0.05,

    # ── Early stopping per stage ─────────────────────────────────
    'patience': 10,
}

if H['debug']:
    H['stages']       = [(64, 2, 4, 1), (128, 2, 2, 1), (256, 2, 1, 1)]
    H['ndf']          = 16
    H['ngf']          = 16
    H['n_res_blocks'] = {64: 1, 128: 1, 256: 1}

torch.backends.cudnn.benchmark = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ─────────────────────────────────────────────────────────────────────
#  DATA
# ─────────────────────────────────────────────────────────────────────
ROOT          = '/kaggle/input/datasets/noob786/p2pgan-human2comic-face-paired-dataset/paired_dataset'
METADATA_PATH = f'{ROOT}/metadata.csv'

df       = pd.read_csv(METADATA_PATH)
if H['debug']:
    df = df.iloc[:40]
train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df   = df[df['split'] == 'val'  ].reset_index(drop=True)
print(f"Train: {len(train_df)}, Val: {len(val_df)}")


def make_transform(size, augment=False):
    ops = [transforms.Resize((size, size),
                              interpolation=transforms.InterpolationMode.LANCZOS)]
    if augment:
        ops += [
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.1, contrast=0.1,
                                   saturation=0.1, hue=0.05),
        ]
    ops += [transforms.ToTensor(),
            transforms.Normalize((0.5,)*3, (0.5,)*3)]
    return transforms.Compose(ops)


class HumanToComicDataset(Dataset):
    def __init__(self, root, df, size=256, augment=False):
        self.paths     = [f'{root}/{p}' for p in df['image_path']]
        self.transform = make_transform(size, augment=augment)
        # Paired flip state is shared between human/comic by using same seed
        self.augment   = augment

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        try:
            img   = np.array(Image.open(self.paths[idx]).convert('RGB'))
            seed  = random.randint(0, 2**32)
            # Apply identical random transforms to both halves
            random.seed(seed); torch.manual_seed(seed)
            human = self.transform(Image.fromarray(img[:, :256]))
            random.seed(seed); torch.manual_seed(seed)
            comic = self.transform(Image.fromarray(img[:, 256:]))
            return human, comic
        except Exception as e:
            print(f"[WARN] {self.paths[idx]}: {e}")
            z = torch.zeros(3, 256, 256)
            return z, z


def make_loaders(size, batch_size, num_workers=2):
    kw = dict(num_workers=num_workers, pin_memory=True,
              persistent_workers=(num_workers > 0))
    tr = DataLoader(HumanToComicDataset(ROOT, train_df, size, augment=True),
                    batch_size=batch_size, shuffle=True,  **kw)
    vl = DataLoader(HumanToComicDataset(ROOT, val_df,   size, augment=False),
                    batch_size=batch_size, shuffle=False, **kw)
    return tr, vl


def denormalize(t):
    return ((t.cpu() * 0.5 + 0.5).clamp(0, 1)
              .permute(1, 2, 0).numpy() * 255).astype(np.uint8)


# ─────────────────────────────────────────────────────────────────────
#  DIFF-AUGMENT  (color + translation + cutout)
#  Applied identically to real and fake before D forward pass.
#  Critically: NOT applied to G's reconstruction losses.
# ─────────────────────────────────────────────────────────────────────
def rand_brightness(x):
    x = x + (torch.rand(x.size(0), 1, 1, 1, device=x.device) - 0.5)
    return x

def rand_saturation(x):
    x_mean = x.mean(dim=1, keepdim=True)
    x = (x - x_mean) * (torch.rand(x.size(0), 1, 1, 1, device=x.device) * 2) + x_mean
    return x

def rand_contrast(x):
    x_mean = x.mean(dim=[1,2,3], keepdim=True)
    x = (x - x_mean) * (torch.rand(x.size(0), 1, 1, 1, device=x.device) + 0.5) + x_mean
    return x

def rand_translation(x, ratio=0.125):
    shift_x = int(x.size(3) * ratio + 0.5)
    shift_y = int(x.size(2) * ratio + 0.5)
    tx = random.randint(-shift_x, shift_x)
    ty = random.randint(-shift_y, shift_y)
    return x.roll(tx, dims=3).roll(ty, dims=2)

def rand_cutout(x, ratio=0.5):
    cut = int(x.size(2) * ratio + 0.5)
    ox = random.randint(0, x.size(2) - cut)
    oy = random.randint(0, x.size(3) - cut)
    x = x.clone()
    x[:, :, oy:oy+cut, ox:ox+cut] = 0
    return x

def diff_augment(x, policy='color,translation,cutout'):
    for p in policy.split(','):
        if p == 'color':
            x = rand_brightness(x)
            x = rand_saturation(x)
            x = rand_contrast(x)
        elif p == 'translation':
            x = rand_translation(x)
        elif p == 'cutout':
            x = rand_cutout(x)
    return x.contiguous()


# ─────────────────────────────────────────────────────────────────────
#  NORMALIZATION  — GroupNorm works at any spatial size (incl. 1×1)
# ─────────────────────────────────────────────────────────────────────
def norm(c):
    for g in [32, 16, 8, 4, 2, 1]:
        if c % g == 0:
            return nn.GroupNorm(g, c)

SN = nn.utils.spectral_norm


# ─────────────────────────────────────────────────────────────────────
#  ATTENTION
# ─────────────────────────────────────────────────────────────────────
class ChannelAttention(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        mid = max(c // r, 4)
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.mx  = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(c, mid, 1, bias=False), nn.ReLU(),
            nn.Conv2d(mid, c, 1, bias=False))
    def forward(self, x):
        return x * torch.sigmoid(self.mlp(self.avg(x)) + self.mlp(self.mx(x)))


class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, 7, padding=3, bias=False)
    def forward(self, x):
        return x * torch.sigmoid(
            self.conv(torch.cat([x.mean(1, keepdim=True),
                                 x.max(1, keepdim=True)[0]], 1)))


class CBAM(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.ca = ChannelAttention(c)
        self.sa = SpatialAttention()
    def forward(self, x): return self.sa(self.ca(x))


class SelfAttention(nn.Module):
    """fp32 softmax + clamped logits — numerically stable under AMP."""
    def __init__(self, c):
        super().__init__()
        mid = max(c // 8, 1)
        self.q = SN(nn.Conv2d(c, mid, 1))
        self.k = SN(nn.Conv2d(c, mid, 1))
        self.v = SN(nn.Conv2d(c, c,   1))
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        B, C, H, W = x.shape
        q = self.q(x).view(B, -1, H*W).permute(0, 2, 1).float()
        k = self.k(x).view(B, -1, H*W).float()
        v = self.v(x).view(B, -1, H*W).float()
        attn = F.softmax(
            (torch.bmm(q, k) * (q.shape[-1] ** -0.5)).clamp(-10, 10), dim=-1)
        out  = torch.bmm(v, attn.permute(0, 2, 1)).view(B, C, H, W)
        return (x + self.gamma * out.to(x.dtype)).contiguous()


# ─────────────────────────────────────────────────────────────────────
#  RESIDUAL BLOCK
# ─────────────────────────────────────────────────────────────────────
class ResBlock(nn.Module):
    def __init__(self, c, dropout=0.0):
        super().__init__()
        layers = [norm(c), nn.ReLU(),
                  nn.Conv2d(c, c, 3, padding=1, bias=False),
                  norm(c), nn.ReLU()]
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        layers.append(nn.Conv2d(c, c, 3, padding=1, bias=False))
        self.block = nn.Sequential(*layers)
    def forward(self, x): return x + self.block(x)


# ─────────────────────────────────────────────────────────────────────
#  ENCODER / DECODER BLOCKS
# ─────────────────────────────────────────────────────────────────────
class DownBlock(nn.Module):
    def __init__(self, ic, oc, outermost=False):
        super().__init__()
        seq = []
        if not outermost:
            seq.append(nn.LeakyReLU(0.2))
        seq.append(nn.Conv2d(ic, oc, 4, stride=2, padding=1, bias=outermost))
        if not outermost:
            seq.append(norm(oc))
        self.block = nn.Sequential(*seq)
    def forward(self, x): return self.block(x)


class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch, outermost=False, dropout=0.0):
        super().__init__()
        seq = [nn.ReLU(),
               nn.ConvTranspose2d(in_ch, out_ch, 4, stride=2, padding=1)]
        if outermost:
            seq.append(nn.Tanh())
        else:
            seq.append(norm(out_ch))
            if dropout > 0:
                seq.append(nn.Dropout(dropout))
        self.block = nn.Sequential(*seq)
    def forward(self, x): return self.block(x)


# ─────────────────────────────────────────────────────────────────────
#  GENERATOR  (same U-Net structure as v4 + SA at enc3 + per-stage bottleneck)
# ─────────────────────────────────────────────────────────────────────
class HumanToComicGenerator(nn.Module):

    def __init__(self, input_nc=3, output_nc=3, ngf=64, dropout=0.3):
        super().__init__()
        nf = ngf
        self._nf      = nf
        self._dropout = dropout

        # ── Encoder ────────────────────────────────────────────────
        self.enc1 = DownBlock(input_nc, nf,    outermost=True)
        self.enc2 = DownBlock(nf,       nf*2)
        self.enc3 = DownBlock(nf*2,     nf*4)
        self.enc4 = DownBlock(nf*4,     nf*8)
        self.enc5 = DownBlock(nf*8,     nf*8)
        self.enc6 = DownBlock(nf*8,     nf*8)

        # ── SA at enc3 level (nf*4) — NEW ─────────────────────────
        self.sa_enc3 = SelfAttention(nf*4)

        # ── Bottleneck — rebuilt by set_depth() ───────────────────
        self.bottleneck = nn.Identity()   # placeholder

        # ── CBAM skip gates ────────────────────────────────────────
        self.cbam1 = CBAM(nf)
        self.cbam2 = CBAM(nf*2)
        self.cbam3 = CBAM(nf*4)
        self.cbam4 = CBAM(nf*8)
        self.cbam5 = CBAM(nf*8)

        # ── Decoder blocks ─────────────────────────────────────────
        self.dec6_ns = UpBlock(nf*8,        nf*8, dropout=dropout)
        self.dec5_ns = UpBlock(nf*8,        nf*8, dropout=dropout)
        self.dec4_ns = UpBlock(nf*8,        nf*4)
        self.dec5    = UpBlock(nf*8 + nf*8, nf*8, dropout=dropout)
        self.dec4    = UpBlock(nf*8 + nf*8, nf*4)
        self.dec3    = UpBlock(nf*4 + nf*4, nf*2)
        self.dec2    = UpBlock(nf*2 + nf*2, nf)
        self.dec1    = UpBlock(nf   + nf,   output_nc, outermost=True)

        self.depth = 6

    # ── Rebuild bottleneck with correct #res-blocks per stage ─────
    def set_depth(self, resolution: int, n_res_blocks: int = 6):
        nf      = self._nf
        self.depth = int(math.log2(resolution)) - 2
        bot_sz  = resolution // (2 ** self.depth)
        res_blocks = [ResBlock(nf*8, self._dropout) for _ in range(n_res_blocks)]
        res_blocks.append(SelfAttention(nf*8))
        self.bottleneck = nn.Sequential(*res_blocks).to(
            next(self.parameters()).device)
        print(f"  Generator depth={self.depth}  "
              f"(input={resolution}px, bottleneck={bot_sz}×{bot_sz}, "
              f"res_blocks={n_res_blocks})")

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.sa_enc3(self.enc3(e2))   # SA applied after enc3
        e4 = self.enc4(e3)
        e5 = self.enc5(e4) if self.depth >= 5 else None
        e6 = self.enc6(e5) if self.depth >= 6 else None

        bot_in = {6: e6, 5: e5, 4: e4}[self.depth]
        b = self.bottleneck(bot_in)

        if self.depth == 6:
            d = self.dec6_ns(b)
            d = self.dec5(torch.cat([d, self.cbam5(e5)], 1).contiguous())
            d = self.dec4(torch.cat([d, self.cbam4(e4)], 1).contiguous())
        elif self.depth == 5:
            d = self.dec5_ns(b)
            d = self.dec4(torch.cat([d, self.cbam4(e4)], 1).contiguous())
        else:
            d = self.dec4_ns(b)

        d = self.dec3(torch.cat([d, self.cbam3(e3)], 1).contiguous())
        d = self.dec2(torch.cat([d, self.cbam2(e2)], 1).contiguous())
        d = self.dec1(torch.cat([d, self.cbam1(e1)], 1).contiguous())
        return d


# ─────────────────────────────────────────────────────────────────────
#  PATCH DISCRIMINATOR  (unchanged from v4)
# ─────────────────────────────────────────────────────────────────────
class PatchDiscriminator(nn.Module):
    def __init__(self, input_nc=6, ndf=64, n_layers=3):
        super().__init__()
        kw  = 4
        seq = [SN(nn.Conv2d(input_nc, ndf, kw, stride=2, padding=1)),
               nn.LeakyReLU(0.2, inplace=True)]
        nf  = ndf
        for _ in range(1, n_layers):
            nf_p, nf = nf, min(nf*2, 512)
            seq += [SN(nn.Conv2d(nf_p, nf, kw, stride=2, padding=1)),
                    norm(nf), nn.LeakyReLU(0.2, inplace=True)]
        nf_p, nf = nf, min(nf*2, 512)
        seq += [SN(nn.Conv2d(nf_p, nf, kw, stride=1, padding=1)),
                norm(nf), nn.LeakyReLU(0.2, inplace=True),
                SN(nn.Conv2d(nf, 1, kw, stride=1, padding=1))]
        self.layers = nn.ModuleList(seq)

    def forward(self, x):
        feats = []
        for layer in self.layers:
            x = layer(x)
            feats.append(x)
        return feats[-1], feats[:-1]


# ─────────────────────────────────────────────────────────────────────
#  VGG PERCEPTUAL LOSS  (fp32 forced — safe under AMP)
# ─────────────────────────────────────────────────────────────────────
class VGGPerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg      = models.vgg16(weights=models.VGG16_Weights.DEFAULT).features
        self.sl1 = nn.Sequential(*list(vgg)[:4])
        self.sl2 = nn.Sequential(*list(vgg)[4:9])
        self.sl3 = nn.Sequential(*list(vgg)[9:16])
        for p in self.parameters():
            p.requires_grad = False
        self.register_buffer('mean', torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))
        self.register_buffer('std',  torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))

    def _prep(self, x):
        return ((x.float() * 0.5 + 0.5) - self.mean.float()) / self.std.float()

    def forward(self, fake, real):
        with torch.cuda.amp.autocast(enabled=False):
            f, r = self._prep(fake), self._prep(real)
            loss  = 0.0
            for sl in (self.sl1, self.sl2, self.sl3):
                f, r = sl(f), sl(r)
                loss += F.l1_loss(f, r.detach())
        return loss


# ─────────────────────────────────────────────────────────────────────
#  FREQUENCY LOSS  — NEW in v6
#  Penalizes differences in FFT magnitude spectrum.
#  Encourages sharp comic-style edges and hatching patterns.
# ─────────────────────────────────────────────────────────────────────
class FrequencyLoss(nn.Module):
    """L1 on log(1 + |FFT magnitude|) across all channels."""
    def forward(self, fake, real):
        with torch.cuda.amp.autocast(enabled=False):
            f = fake.float()
            r = real.float()
            # Per-channel FFT, shift DC to centre, log-magnitude
            f_mag = torch.fft.fftshift(torch.fft.fft2(f)).abs().add(1).log()
            r_mag = torch.fft.fftshift(torch.fft.fft2(r)).abs().add(1).log()
        return F.l1_loss(f_mag, r_mag.detach())


# ─────────────────────────────────────────────────────────────────────
#  EMA
# ─────────────────────────────────────────────────────────────────────
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay  = decay
        self.shadow = copy.deepcopy(model).eval()
        for p in self.shadow.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def update(self, model):
        for s, m in zip(self.shadow.parameters(), model.parameters()):
            s.data.mul_(self.decay).add_(m.data, alpha=1 - self.decay)

    @torch.no_grad()
    def __call__(self, x): return self.shadow(x)


# ─────────────────────────────────────────────────────────────────────
#  LOSS FUNCTIONS
# ─────────────────────────────────────────────────────────────────────
def smooth(pred, real, rv=0.9, fv=0.1):
    return torch.full_like(pred, rv if real else fv)


def add_noise(x, sigma):
    """Instance noise on D inputs — annealed over training."""
    if sigma > 0:
        return x + sigma * torch.randn_like(x)
    return x


def d_loss_fn(disc, gen, target, src, noise_sigma=0.0,
              rv=0.9, fv=0.1, augment=True):
    with torch.no_grad():
        fake = gen(src)

    real_pair = torch.cat([target, src], 1)
    fake_pair = torch.cat([fake.detach(), src], 1)

    # DiffAugment on D inputs
    if augment:
        real_pair = diff_augment(real_pair)
        fake_pair = diff_augment(fake_pair)

    # Instance noise
    real_pair = add_noise(real_pair, noise_sigma)
    fake_pair = add_noise(fake_pair, noise_sigma)

    rp, _ = disc(real_pair)
    fp, _ = disc(fake_pair)
    return 0.5 * (
        F.binary_cross_entropy_with_logits(rp, smooth(rp, True,  rv, fv)) +
        F.binary_cross_entropy_with_logits(fp, smooth(fp, False, rv, fv)))


def g_loss_fn(disc, gen, target, src,
              vgg_fn, freq_fn,
              lam_l1, lam_perc, lam_fm, lam_freq,
              augment=True):
    fake = gen(src)

    fake_pair = torch.cat([fake, src], 1)
    real_pair = torch.cat([target, src], 1)

    if augment:
        fake_pair_aug = diff_augment(fake_pair)
        real_pair_aug = diff_augment(real_pair)
    else:
        fake_pair_aug = fake_pair
        real_pair_aug = real_pair

    fp, ff = disc(fake_pair_aug)
    with torch.no_grad():
        _,  rf = disc(real_pair_aug)

    adv  = F.binary_cross_entropy_with_logits(fp, torch.ones_like(fp))
    fm   = sum(F.l1_loss(a, b.detach()) for a, b in zip(ff, rf))
    l1   = F.l1_loss(fake, target)
    perc = vgg_fn(fake, target)
    freq = freq_fn(fake, target)

    total = adv + lam_l1*l1 + lam_perc*perc + lam_fm*fm + lam_freq*freq
    return total, {
        'adv':  adv.item(),
        'l1':   l1.item(),
        'perc': perc.item(),
        'fm':   fm.item(),
        'freq': freq.item(),
    }


def has_nan(t): return not torch.isfinite(t)


# ─────────────────────────────────────────────────────────────────────
#  ADAPTIVE D CONTROL
# ─────────────────────────────────────────────────────────────────────
class AdaptiveDControl:
    """
    Tracks a running average of recent D losses and
    returns whether to skip D or run extra D steps.
    """
    def __init__(self, low=0.35, high=0.70, window=50):
        self.low    = low
        self.high   = high
        self.buf    = []
        self.window = window

    def record(self, d_loss):
        self.buf.append(d_loss)
        if len(self.buf) > self.window:
            self.buf.pop(0)

    @property
    def avg(self):
        return sum(self.buf) / max(len(self.buf), 1)

    def d_steps(self):
        """How many D updates to run this iteration."""
        a = self.avg
        if a < self.low:   return 0   # D too strong → skip
        if a > self.high:  return 2   # D too weak   → double
        return 1                       # balanced      → normal


# ─────────────────────────────────────────────────────────────────────
#  TRAIN / EVAL
# ─────────────────────────────────────────────────────────────────────
def train_epoch(disc, gen, ema, loader,
                d_opt, g_opt, d_sch, g_sch,
                d_scaler, g_scaler, device,
                adaptive_d, noise_sigma, accum_steps=1):
    disc.train(); gen.train()
    d_sum = g_sum = skipped = 0
    g_opt.zero_grad(set_to_none=True)

    for batch_idx, (human, comic) in enumerate(tqdm(loader, desc='  train', leave=False)):
        human, comic = human.to(device), comic.to(device)
        is_accum_boundary = ((batch_idx + 1) % accum_steps == 0 or
                              batch_idx == len(loader) - 1)
        scale = 1.0 / accum_steps

        # ── Discriminator ──────────────────────────────────────────
        n_d_steps = adaptive_d.d_steps()
        for _ in range(max(n_d_steps, 1)):   # always run at least 1 to log loss
            d_opt.zero_grad(set_to_none=True)
            with autocast():
                d_loss = d_loss_fn(disc, gen, comic, human,
                                   noise_sigma=noise_sigma,
                                   rv=H['real_label'], fv=H['fake_label'])
            if has_nan(d_loss):
                skipped += 1; continue
            if n_d_steps > 0:
                d_scaler.scale(d_loss).backward()
                d_scaler.unscale_(d_opt)
                nn.utils.clip_grad_norm_(disc.parameters(), H['max_grad_norm'])
                d_scaler.step(d_opt); d_scaler.update()

        adaptive_d.record(d_loss.item() if torch.isfinite(d_loss) else 0.5)

        if n_d_steps == 0:   # D too strong → skip D update this step
            pass

        # ── Generator (with gradient accumulation) ─────────────────
        disc.eval()
        with autocast():
            g_loss, comps = g_loss_fn(
                disc, gen, comic, human, vgg_loss_fn, freq_loss_fn,
                H['lambda_l1'], H['lambda_perc'], H['lambda_fm'], H['lambda_freq'])
        disc.train()
        if has_nan(g_loss):
            skipped += 1
            g_opt.zero_grad(set_to_none=True)
            g_scaler.update()
            continue

        g_scaler.scale(g_loss * scale).backward()

        if is_accum_boundary:
            g_scaler.unscale_(g_opt)
            nn.utils.clip_grad_norm_(gen.parameters(), H['max_grad_norm'])
            g_scaler.step(g_opt); g_scaler.update()
            g_opt.zero_grad(set_to_none=True)
            d_sch.step(); g_sch.step()

        ema.update(gen)
        d_sum += d_loss.item() if torch.isfinite(d_loss) else 0.0
        g_sum += g_loss.item()

    n = max(len(loader) - skipped, 1)
    return d_sum/n, g_sum/n, skipped


@torch.no_grad()
def eval_epoch(disc, gen, loader, device):
    disc.eval(); gen.eval()
    d_sum = g_sum = 0.0
    for human, comic in tqdm(loader, desc='  val  ', leave=False):
        human, comic = human.to(device), comic.to(device)
        with autocast():
            dl = d_loss_fn(disc, gen, comic, human, noise_sigma=0.0,
                           rv=H['real_label'], fv=H['fake_label'], augment=False)
            gl, _ = g_loss_fn(disc, gen, comic, human, vgg_loss_fn, freq_loss_fn,
                              H['lambda_l1'], H['lambda_perc'],
                              H['lambda_fm'],  H['lambda_freq'], augment=False)
        if torch.isfinite(dl): d_sum += dl.item()
        if torch.isfinite(gl): g_sum += gl.item()
    return d_sum/len(loader), g_sum/len(loader)


@torch.no_grad()
def visualize(gen_fn, loader, device, title='', n=4, save_path=None):
    if hasattr(gen_fn, 'eval'): gen_fn.eval()
    fig, axs = plt.subplots(n, 3, figsize=(12, n*4))
    for i, (human, comic) in enumerate(loader):
        if i >= n: break
        h = human[:1].to(device)
        f = gen_fn(h).cpu()
        for col, (img, lbl) in enumerate([
            (h[0].cpu(), 'Input'), (comic[0], 'Target'), (f[0], 'Generated')]):
            axs[i, col].imshow(denormalize(img))
            axs[i, col].set_title(lbl); axs[i, col].axis('off')
    plt.suptitle(f'Human → Comic  {title}')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120)
        print(f"  Saved visualization: {save_path}")
    plt.close()


# ─────────────────────────────────────────────────────────────────────
#  INSTANTIATE
# ─────────────────────────────────────────────────────────────────────
generator     = HumanToComicGenerator(ngf=H['ngf'], dropout=H['dropout']).to(device)
discriminator = PatchDiscriminator(ndf=H['ndf'], n_layers=H['n_layers_D']).to(device)
vgg_loss_fn   = VGGPerceptualLoss().to(device)
freq_loss_fn  = FrequencyLoss().to(device)
ema_gen       = EMA(generator, decay=H['ema_decay'])

print(f"Generator     = {sum(p.numel() for p in generator.parameters())/1e6:.2f}M params")
print(f"Discriminator = {sum(p.numel() for p in discriminator.parameters())/1e6:.2f}M params")

# ── Sanity-check ─────────────────────────────────────────────────────
print("\nSanity-checking all resolutions:")
for res, n_rb in [(64, H['n_res_blocks'][64]),
                  (128, H['n_res_blocks'][128]),
                  (256, H['n_res_blocks'][256])]:
    generator.set_depth(res, n_res_blocks=n_rb)
    ema_gen.shadow.set_depth(res, n_res_blocks=n_rb)
    with torch.no_grad():
        dummy = torch.zeros(1, 3, res, res, device=device)
        out   = generator(dummy)
        dp, _ = discriminator(torch.cat([out, dummy], 1))
    assert out.shape == (1, 3, res, res), f"G output shape wrong at {res}px"
    # Test frequency loss
    fl = freq_loss_fn(out, dummy)
    assert torch.isfinite(fl), "FrequencyLoss returned NaN"
    print(f"  [{res}px] G={out.shape} ✓  D={dp.shape} ✓  FreqLoss={fl.item():.4f} ✓")
print("All resolution checks passed.\n")


# ─────────────────────────────────────────────────────────────────────
#  PROGRESSIVE TRAINING
# ─────────────────────────────────────────────────────────────────────
all_history = []
best_val_g  = float('inf')
total_steps = sum(len(range(1, n_ep+1)) * (9900 // bs)
                  for _, n_ep, bs, _ in H['stages'])  # approx

for stage_idx, (res, n_epochs, bs, accum_steps) in enumerate(H['stages']):
    print(f"\n{'='*60}")
    print(f"  STAGE {stage_idx+1}/{len(H['stages'])}  "
          f"Res={res}×{res}  Epochs={n_epochs}  Batch={bs}  Accum={accum_steps}")
    print(f"{'='*60}")

    n_res_b = H['n_res_blocks'][res]
    generator.set_depth(res, n_res_blocks=n_res_b)
    ema_gen.shadow.set_depth(res, n_res_blocks=n_res_b)

    train_loader, val_loader = make_loaders(res, bs)
    eff_steps = len(train_loader) // accum_steps
    print(f"  Train={len(train_loader)} batches "
          f"({eff_steps} effective steps), Val={len(val_loader)} batches")

    # TTUR: D_lr = 2 × G_lr for faster D recovery when saturated
    d_opt = optim.Adam(discriminator.parameters(),
                       lr=H['lr']*2, betas=(H['beta1'], 0.999))
    g_opt = optim.Adam(generator.parameters(),
                       lr=H['lr'], betas=(H['beta1'], 0.999),
                       weight_decay=H['weight_decay'])

    total  = eff_steps * n_epochs
    warmup = max(total // 8, 100)
    d_sch  = get_cosine_schedule_with_warmup(d_opt, warmup, total)
    g_sch  = get_cosine_schedule_with_warmup(g_opt, warmup, total)
    d_scaler, g_scaler = GradScaler(), GradScaler()
    adaptive_d = AdaptiveDControl(low=H['d_low_thresh'], high=H['d_high_thresh'])

    # Compute D input noise sigma that anneals to 0 by end of stage
    stage_total_batches = len(train_loader) * n_epochs
    patience_counter = 0
    stage_best_val_g = float('inf')

    for epoch in range(1, n_epochs + 1):
        # Anneal noise: full σ for first 10% of stage, linear decay to 0
        noise_sigma = H['d_noise_sigma'] * max(0.0,
            1.0 - (epoch - 1) / max(n_epochs * 0.9, 1))

        tr_d, tr_g, skipped = train_epoch(
            discriminator, generator, ema_gen,
            train_loader, d_opt, g_opt, d_sch, g_sch,
            d_scaler, g_scaler, device,
            adaptive_d=adaptive_d,
            noise_sigma=noise_sigma,
            accum_steps=accum_steps)
        vl_d, vl_g = eval_epoch(discriminator, generator, val_loader, device)

        all_history.append(dict(stage=stage_idx+1, res=res, epoch=epoch,
                                tr_d=tr_d, tr_g=tr_g, vl_d=vl_d, vl_g=vl_g,
                                skipped=skipped, d_avg=adaptive_d.avg,
                                noise_sigma=noise_sigma))
        print(f"  [S{stage_idx+1}] Ep {epoch:3d}/{n_epochs} | "
              f"Train D={tr_d:.4f} G={tr_g:.4f} | "
              f"Val D={vl_d:.4f} G={vl_g:.4f} | "
              f"D_avg={adaptive_d.avg:.4f} σ={noise_sigma:.4f} skip={skipped}")

        # Global best
        if vl_g < best_val_g:
            best_val_g = vl_g
            torch.save(generator.state_dict(),
                       os.path.join(H['save_dir'], 'generator_best.pth'))
            torch.save(ema_gen.shadow.state_dict(),
                       os.path.join(H['save_dir'], 'generator_ema_best.pth'))
            print(f"    ✓ Best val G={vl_g:.4f} — saved")

        # Stage early stopping
        if vl_g < stage_best_val_g:
            stage_best_val_g = vl_g
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= H['patience']:
                print(f"    → Early stopping at epoch {epoch} "
                      f"(no improvement for {H['patience']} epochs)")
                break

        torch.cuda.empty_cache()

    print(f"\n  Visualising stage {stage_idx+1} (EMA generator):")
    visualize(ema_gen, val_loader, device,
              title=f'Stage {stage_idx+1} — {res}×{res}',
              save_path=os.path.join(H['save_dir'],
                                     f'vis_stage{stage_idx+1}_{res}px.png'))

# ─────────────────────────────────────────────────────────────────────
#  FINAL SAVE
# ─────────────────────────────────────────────────────────────────────
torch.save(discriminator.state_dict(),
           os.path.join(H['save_dir'], 'discriminator_final.pth'))
torch.save(generator.state_dict(),
           os.path.join(H['save_dir'], 'generator_final.pth'))
torch.save(ema_gen.shadow.state_dict(),
           os.path.join(H['save_dir'], 'generator_ema_final.pth'))
print("\nAll models saved.")

# ─────────────────────────────────────────────────────────────────────
#  LOSS CURVES
# ─────────────────────────────────────────────────────────────────────
hist = pd.DataFrame(all_history)
fig, axes = plt.subplots(1, len(H['stages']), figsize=(7*len(H['stages']), 5))
if len(H['stages']) == 1: axes = [axes]
for ax, (si, (res, *_)) in zip(axes, enumerate(H['stages'])):
    s = hist[hist['stage'] == si+1]
    ax.plot(s['epoch'], s['tr_d'], label='Train D', color='steelblue')
    ax.plot(s['epoch'], s['tr_g'], label='Train G', color='darkorange')
    ax.plot(s['epoch'], s['vl_d'], label='Val D',
            color='steelblue',   linestyle='--')
    ax.plot(s['epoch'], s['vl_g'], label='Val G',
            color='darkorange', linestyle='--')
    ax.set_title(f'Stage {si+1} — {res}×{res}')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(); ax.grid(True, alpha=0.4)
plt.suptitle('Human → Comic Progressive Losses v6', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(H['save_dir'], 'loss_curves_v6.png'), dpi=120)
plt.close()
print("Loss curves saved.")

Using device: cuda
Train: 9900, Val: 100


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100%|██████████| 528M/528M [00:02<00:00, 240MB/s]


Generator     = 35.70M params
Discriminator = 2.77M params

Sanity-checking all resolutions:
  Generator depth=4  (input=64px, bottleneck=4×4, res_blocks=2)
  Generator depth=4  (input=64px, bottleneck=4×4, res_blocks=2)
  [64px] G=torch.Size([1, 3, 64, 64]) ✓  D=torch.Size([1, 1, 6, 6]) ✓  FreqLoss=2.8292 ✓
  Generator depth=5  (input=128px, bottleneck=4×4, res_blocks=4)
  Generator depth=5  (input=128px, bottleneck=4×4, res_blocks=4)
  [128px] G=torch.Size([1, 3, 128, 128]) ✓  D=torch.Size([1, 1, 14, 14]) ✓  FreqLoss=3.5261 ✓
  Generator depth=6  (input=256px, bottleneck=4×4, res_blocks=6)
  Generator depth=6  (input=256px, bottleneck=4×4, res_blocks=6)
  [256px] G=torch.Size([1, 3, 256, 256]) ✓  D=torch.Size([1, 1, 30, 30]) ✓  FreqLoss=4.1899 ✓
All resolution checks passed.


  STAGE 1/3  Res=64×64  Epochs=50  Batch=8  Accum=1
  Generator depth=4  (input=64px, bottleneck=4×4, res_blocks=2)
  Generator depth=4  (input=64px, bottleneck=4×4, res_blocks=2)
  Train=1238 batches (1238 eff

  [S1] Ep   1/50 | Train D=0.5628 G=39.5752 | Val D=0.4647 G=31.4334 | D_avg=0.5881 σ=0.0500 skip=0
    ✓ Best val G=31.4334 — saved


  [S1] Ep   2/50 | Train D=0.5656 G=33.7128 | Val D=0.4998 G=31.4155 | D_avg=0.5752 σ=0.0489 skip=0
    ✓ Best val G=31.4155 — saved


...

  [S3] Ep  16/20 | Train D=0.4889 G=20.0052 | Val D=0.4606 G=17.9754 | D_avg=0.4736 σ=0.0083 skip=0
    ✓ Best val G=17.9754 — saved


  [S3] Ep  17/20 | Train D=0.4814 G=19.9357 | Val D=0.4859 G=17.8843 | D_avg=0.4566 σ=0.0056 skip=0
    ✓ Best val G=17.8843 — saved


  [S3] Ep  18/20 | Train D=0.4714 G=19.8913 | Val D=0.4528 G=17.9506 | D_avg=0.4985 σ=0.0028 skip=0


  [S3] Ep  19/20 | Train D=0.4677 G=19.8113 | Val D=0.4544 G=17.8349 | D_avg=0.4699 σ=0.0000 skip=0
    ✓ Best val G=17.8349 — saved


  [S3] Ep  20/20 | Train D=0.4671 G=19.7969 | Val D=0.4460 G=17.8964 | D_avg=0.4827 σ=0.0000 skip=0

  Visualising stage 3 (EMA generator):
  Saved visualization: ./vis_stage3_256px.png

All models saved.
Loss curves saved.
